<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/5/58/Uber_logo_2018.svg/1024px-Uber_logo_2018.svg.png" alt="UBER LOGO" width="50%" />

# UBER Pickups 

# import libraries

In [1]:
import pandas as pd
import plotly.express as px

# Read data

In [9]:
location = pd.read_csv('c/location.csv')
print(f'location.csv : {location.shape[0]} lines / {location.shape[1]} columns')
pickup_14 = pd.read_csv('c/2014.csv')
print(f'2014.csv : {pickup_14.shape[0]} lines / {pickup_14.shape[1]} columns')
pickup = pd.read_csv('c/2015.csv')
print(f'2015.csv : {pickup.shape[0]} lines / {pickup.shape[1]} columns')

location.csv : 265 lines / 3 columns
2014.csv : 4534327 lines / 4 columns
2015.csv : 14270479 lines / 8 columns


# Analyze taxi_zone_lookup.csv

In [8]:
mask = location['locationid'].value_counts() == 1
print(mask.all()) # True attendu si toutes les valeurs sont uniques

True


In [ ]:
location['borough'].value_counts()

borough
queens           69
manhattan        69
brooklyn         61
bronx            43
staten island    20
unknown           2
ewr               1
Name: count, dtype: int64

In [ ]:
location['zone'].value_counts()

zone
governor's island/ellis island/liberty island    3
corona                                           2
unknown                                          2
newark airport                                   1
jamaica bay                                      1
                                                ..
woodlawn/wakefield                               1
woodside                                         1
world trade center                               1
yorkville east                                   1
yorkville west                                   1
Name: count, Length: 261, dtype: int64

In [ ]:
location[location.duplicated("zone", keep=False)]  # Corona (56, 57), Governor's Island (103 à 105), Unknown (264, 265)

,locationid,borough,zone
55,56,queens,corona
56,57,queens,corona
102,103,manhattan,governor's island/ellis island/liberty island
103,104,manhattan,governor's island/ellis island/liberty island
104,105,manhattan,governor's island/ellis island/liberty island
263,264,unknown,unknown
264,265,unknown,unknown


Borough est beaucoup moins précis que zone pour tracer les pickups. 
We observe :
- zone de corona dans le queens en doublon. 
- 3 références communes sur les 3 îlots sur l'Hudson.
- deux zones sans étiquette.

# Analyze raw_2015

Le fichier trace les pickups sur 6 mois d'exploitation en 2015 de janvier à juin. Il contient la datetime, les références du dispatch, taxi et emplacement. 
We observe missing values sur le dispatch 1,14 %.

# Identify base in 2015

In [ ]:
# pickup_14['base'].isin(pickup['dispatching_base_num']).value_counts()

On retrouve le nombre de lignes exact de raw_2014. Aucune base ne diffère des dispatching_base_num.
Pour s'en assurer, comparons les valeurs uniques.

In [ ]:
b14 = set(pickup_14['base'].unique())
b15_dispatch = set(pickup['dispatching_base_num'].unique())
b15_licence = set(pickup['affiliated_base_num'].unique())
print(len(b14), len(b15_dispatch), len(b15_licence), len(b14 & b15_dispatch & b15_licence))

Base correspond à la référence du centre de dispatch. De 2014 à 2015, le nombre des centres de dispatch a été augmenté de 3.

# Identify location pickup

locationid est présent sur uber-raw-data-janjune-15 et taxi-zone-lookup. Nous pouvons les fusionner à partir de cette colonne.

# Analyze pickup_14

Les fichiers tracent 6 mois d'exploitation en 2014 d'avril à septembre. Les informations diffèrent : date/time, lat, lon, base.
Cette fois, nous disposons des coordonnées des pickups.
Nous devons identifier les emplacements et établir si base est la référence du dispatch ou du taxi. 

In [ ]:
pickup_14_volume = pickup_14["base"].value_counts()
pickup_volume = pickup["dispatching_base_num"].value_counts()
print(pickup_14_volume, pickup_volume)

In [ ]:
# import pandas as pd
# import matplotlib.pyplot as plt

# v14 = raw_2014["base"].value_counts().rename("2014")
# v15 = raw_2015["dispatching_base_num"].value_counts().rename("2015")

# pd.concat([v14, v15], axis=1).plot(kind="bar", figsize=(8, 5))
# plt.xlabel("dispatch_base")
# plt.ylabel("pickups")
# plt.show()

In [ ]:
v = pd.concat([pickup_14["base"].value_counts().rename("2014"),
               pickup["dispatching_base_num"].value_counts().rename("2015")], axis=1).fillna(0)
v.index.name = "base"

fig = px.bar(v.reset_index(), x="base", y=["2014", "2015"], barmode="group",
             title="Evolution of volume pickups (millions) by dispatch base on 6 months",
             labels={"value": "Pickups", "variable": "Year"})
fig.update_layout(hovermode="x unified")
fig.show()

# Analyze one hour on one day

In [ ]:
pickup.head(1)

In [ ]:
pickup_14['date/time'].dtype

In [ ]:
pickup['pickup_date'].dtype

In [ ]:
begin = pd.Timestamp('2014-04-01 00:00:00')
end = begin + pd.Timedelta(hours=1)

t1 = pickup_14[(pickup_14['date/time'] >= begin) & (pickup_14['date/time'] < end)]

In [ ]:
t1

In [ ]:
bases_median = t1.groupby('base')[['lat', 'lon']].median().reset_index()

In [ ]:
print(bases_median)

In [ ]:
fig = px.scatter_map(bases_median, lat="lat", lon="lon", color="base", zoom=10, map_style="carto-positron")
fig.show()

Les bases b02617 et b02764 se superposent. Let's record bases and search for duplicated values.

# Create pickup

In [ ]:
bases = pickup_14.groupby('base')[['lat', 'lon']].median().reset_index()

In [ ]:
pickup_14[['base', 'lon', 'lat']].to_csv("data/bases.csv", index=False)

In [ ]:
print(bases.head(1))

# Search for duplicated values

In [ ]:
duplicated_values = bases[bases.duplicated(subset=["lat", "lon"], keep=False)]
print(duplicated_values.sort_values(['base', "lat", "lon"]))

In [ ]:
with pd.option_context("display.precision", 15):
    print(bases_median)

There are no duplicated values, only values very similar. Des points proches signalent justement une forte densité de pickups, donc une hot-zone. Les supprimer ferait perdre cette information et fausserait les clusters (KMeans comme DBSCAN).

In [ ]:
# bases_t1 = bases_median.round({"lat": 4, "lon": 4})
# print(bases_t1[bases_t1.duplicated(subset=["lat", "lon"], keep=False)])

# Merge raw_2015 et bases

In [ ]:
pickup = pickup.merge(bases[["base", "lon", "lat"]], left_on="dispatching_base_num", right_on="base", how="left").drop(columns="base")
print(pickup[["dispatching_base_num", "lon", "lat"]])
print(pickup[["lon", "lat"]])

In [ ]:
pickup.head(1)

Vérifier l'utilité de locationid.

# Detect outlyers

On cherche seulement les points aberrants. Dans l'eau ou en dehors de l'état de new york. Sur sample, il n'y en a pas.

# Create sample

In [ ]:
mask = pickup[pickup['dispatching_base_num'].isin(bases_median['base'])]
n = 5000  # taille cible, choix provisoire 5000 puis 2500
frac = n / len(mask)
sample = mask.groupby('dispatching_base_num').sample(frac=frac, random_state=42)
print(frac, len(sample))

In [ ]:
sample.to_csv("data/sample.csv", index=False)